### 2D and 3D phase space reconstructions

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.neighbors import NearestNeighbors

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"
eeg_path = os.path.join(base_dir, "eeg_data_with_channels.npy")

embedding_2d_dir = os.path.join(base_dir, "embedding_data", "2dembedding_data")
embedding_3d_dir = os.path.join(base_dir, "embedding_data", "3dembedding_data")
plots_directory = os.path.join(base_dir, "plots")

for dir_path in [embedding_2d_dir, embedding_3d_dir, plots_directory]:
    os.makedirs(dir_path, exist_ok=True)

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

sampling_rate = 1000
start_time, end_time = 814.571, 921.515
start_index, end_index = int(start_time * sampling_rate), int(end_time * sampling_rate)

# Delay-estimation settings
max_delay = 20
subsample_factor = 10
n_bins = 64
delay_rule = "first_local_min"   # "first_local_min" or "global_min"

# Embeddings to save
emb_dim_2d = 2
emb_dim_3d = 3

# Optional FNN settings (not required for saving 2D/3D, but useful to keep)
max_dim = 20
fnn_R = 10.0

# Plot speedup: only subsample for visualization, not for saved embeddings
plot_max_points = 15000

# Optional plot styling
ACCENT = "cyan"
BG = "black"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT,
})

rng = np.random.default_rng(0)

# -------------------------------------------------------
# STYLE HELPER
# -------------------------------------------------------
def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, alpha=0.20, color=ACCENT)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)

# -------------------------------------------------------
# LOAD EEG
# -------------------------------------------------------
EEG_data = np.load(eeg_path, allow_pickle=True)

if EEG_data.ndim != 2:
    raise ValueError(f"Expected 2D EEG array, got shape {EEG_data.shape}")

# Handle either (time, channels) or (channels, time)
if EEG_data.shape[1] == len(eeg_channel_names):
    filtered_EEG_data = EEG_data[start_index:end_index, :]
elif EEG_data.shape[0] == len(eeg_channel_names):
    filtered_EEG_data = EEG_data[:, start_index:end_index].T
else:
    raise ValueError(f"Unexpected EEG shape: {EEG_data.shape}")

print(f"Loaded EEG segment: {filtered_EEG_data.shape}")
print(f"Segment: [{start_time}, {end_time}] sec")

# -------------------------------------------------------
# AMI / DELAY ESTIMATION
# -------------------------------------------------------
def standardize_1d(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + 1e-12)

def digitize_equal_width(x, n_bins=64):
    """
    Fixed-width bins after standardization.
    Returns integer bin IDs in [0, n_bins-1].
    """
    x = standardize_1d(x)
    xmin, xmax = np.min(x), np.max(x)

    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
        return np.zeros_like(x, dtype=np.int32)

    edges = np.linspace(xmin, xmax, n_bins + 1)
    # digitize -> 1..n_bins, then shift to 0..n_bins-1
    xb = np.digitize(x, edges[1:-1], right=False).astype(np.int32)
    xb = np.clip(xb, 0, n_bins - 1)
    return xb

def average_mutual_information_from_binned(xb, max_delay=20, n_bins=64):
    """
    Fast AMI using 2D joint histograms via bincount.
    xb must be integer-binned already.
    """
    xb = np.asarray(xb, dtype=np.int32)
    N = len(xb)

    ami = np.full(max_delay, np.nan, dtype=float)

    for tau in range(1, max_delay + 1):
        x1 = xb[:-tau]
        x2 = xb[tau:]

        # joint histogram via linearized indices
        joint_idx = x1 * n_bins + x2
        joint = np.bincount(joint_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins).astype(float)

        total = joint.sum()
        if total <= 0:
            continue

        pxy = joint / total
        px = pxy.sum(axis=1, keepdims=True)
        py = pxy.sum(axis=0, keepdims=True)

        mask = pxy > 0
        ami[tau - 1] = np.sum(pxy[mask] * np.log(pxy[mask] / (px @ py)[mask]))

    return ami

def first_local_minimum(y):
    """
    Return index of first local minimum in y (1-based delay convention handled outside).
    Fallback to global minimum if no local minimum exists.
    """
    y = np.asarray(y, dtype=float)
    for i in range(1, len(y) - 1):
        if np.isfinite(y[i-1]) and np.isfinite(y[i]) and np.isfinite(y[i+1]):
            if y[i] < y[i-1] and y[i] <= y[i+1]:
                return i
    return int(np.nanargmin(y))

def determine_delay(data, max_delay=20, subsample_factor=10, n_bins=64, rule="first_local_min"):
    xs = np.asarray(data, dtype=float)[::subsample_factor]
    xb = digitize_equal_width(xs, n_bins=n_bins)
    ami = average_mutual_information_from_binned(xb, max_delay=max_delay, n_bins=n_bins)

    if rule == "first_local_min":
        idx = first_local_minimum(ami)
    elif rule == "global_min":
        idx = int(np.nanargmin(ami))
    else:
        raise ValueError(f"Unknown rule: {rule}")

    tau = idx + 1
    return tau, ami

# -------------------------------------------------------
# DELAY EMBEDDING
# -------------------------------------------------------
def delay_embedding(data, emb_dim, delay):
    """
    Fast embedding using sliding_window_view.
    """
    x = np.asarray(data, dtype=float)
    span = (emb_dim - 1) * delay + 1
    M = len(x) - span + 1

    if M <= 0:
        return None

    win = sliding_window_view(x, span)
    emb = win[:, ::delay]
    return np.asarray(emb, dtype=float)

# -------------------------------------------------------
# OPTIONAL: FALSE NEAREST NEIGHBORS
# -------------------------------------------------------
def false_nearest_neighbors(data, max_dim, delay, R=10.0, atol_eps=1e-12):
    """
    Proper FNN estimate across d=1..max_dim-1 using d and d+1 embeddings.
    """
    x = np.asarray(data, dtype=float)
    fnn = np.full(max_dim, np.nan, dtype=float)

    for d in range(1, max_dim):
        Xd = delay_embedding(x, d, delay)
        Xd1 = delay_embedding(x, d + 1, delay)

        if Xd is None or Xd1 is None:
            continue

        M = min(len(Xd), len(Xd1))
        Xd = Xd[:M]
        Xd1 = Xd1[:M]

        nbrs = NearestNeighbors(n_neighbors=2, algorithm="auto").fit(Xd)
        distances, indices = nbrs.kneighbors(Xd)

        nn_idx = indices[:, 1]
        dist_d = distances[:, 1]

        extra_coord_diff = np.abs(Xd1[:, -1] - Xd1[nn_idx, -1])
        ratio = extra_coord_diff / (dist_d + atol_eps)

        fnn[d - 1] = np.mean(ratio > R)

    return fnn

# -------------------------------------------------------
# PLOT HELPERS
# -------------------------------------------------------
def subsample_rows(X, max_points=15000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n = len(X)
    if n <= max_points:
        return X
    idx = np.sort(rng.choice(n, size=max_points, replace=False))
    return X[idx]

# -------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------
tau_rows = []
ami_store = {}

for ch_idx, channel_name in enumerate(eeg_channel_names):
    channel_data = filtered_EEG_data[:, ch_idx].astype(float)

    optimal_delay, ami_values = determine_delay(
        channel_data,
        max_delay=max_delay,
        subsample_factor=subsample_factor,
        n_bins=n_bins,
        rule=delay_rule
    )

    embedded_data_2d = delay_embedding(channel_data, emb_dim_2d, optimal_delay)
    embedded_data_3d = delay_embedding(channel_data, emb_dim_3d, optimal_delay)

    if embedded_data_2d is None or embedded_data_3d is None:
        print(f"Skipping {channel_name}: embedding failed for tau={optimal_delay}")
        continue

    # Save full embeddings
    np.save(os.path.join(embedding_2d_dir, f"2dembedded_{channel_name}.npy"), embedded_data_2d)
    np.save(os.path.join(embedding_3d_dir, f"3dembedded_{channel_name}.npy"), embedded_data_3d)

    tau_rows.append({
        "channel": channel_name,
        "tau": optimal_delay,
        "ami_min": float(np.nanmin(ami_values)),
    })
    ami_store[channel_name] = ami_values

    print(
        f"{channel_name:>3s} | tau={optimal_delay:2d} | "
        f"2D shape={embedded_data_2d.shape} | 3D shape={embedded_data_3d.shape}"
    )

    # -------------------------
    # 2D plot (subsampled only for speed)
    # -------------------------
    X2_plot = subsample_rows(embedded_data_2d, max_points=plot_max_points, rng=rng)

    fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
    style_ax(ax)
    ax.scatter(
        X2_plot[:, 0], X2_plot[:, 1],
        s=1, color=ACCENT
    )
    ax.set_title(f"2D Phase Space Reconstruction | {channel_name} | tau={optimal_delay}")
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_directory, f"2D_{channel_name}.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # -------------------------
    # 3D plot (subsampled only for speed)
    # -------------------------
    X3_plot = subsample_rows(embedded_data_3d, max_points=plot_max_points, rng=rng)

    fig = plt.figure(figsize=(8, 6), facecolor=BG)
    ax = fig.add_subplot(111, projection="3d")
    ax.set_facecolor(BG)
    ax.scatter(
        X3_plot[:, 0], X3_plot[:, 1], X3_plot[:, 2],
        s=1, color=ACCENT
    )
    ax.set_title(f"3D Phase Space Reconstruction | {channel_name} | tau={optimal_delay}", color=ACCENT)
    ax.set_xlabel("Component 1", color=ACCENT)
    ax.set_ylabel("Component 2", color=ACCENT)
    ax.set_zlabel("Component 3", color=ACCENT)
    ax.tick_params(colors=ACCENT)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_directory, f"3D_{channel_name}.png"), dpi=150, bbox_inches="tight")
    plt.close()

# -------------------------------------------------------
# SAVE TAU SUMMARY
# -------------------------------------------------------
tau_df = pd.DataFrame(tau_rows)
tau_csv_path = os.path.join(base_dir, "embedding_data", "taus_2d3d_summary.csv")
tau_npy_path = os.path.join(base_dir, "embedding_data", "taus_2d3d_summary.npy")

os.makedirs(os.path.dirname(tau_csv_path), exist_ok=True)
tau_df.to_csv(tau_csv_path, index=False)
np.save(tau_npy_path, {r["channel"]: r["tau"] for r in tau_rows}, allow_pickle=True)

print(f"\nSaved tau CSV: {tau_csv_path}")
print(f"Saved tau NPY: {tau_npy_path}")

# -------------------------------------------------------
# SAVE AMI CURVES
# -------------------------------------------------------
ami_npz_path = os.path.join(base_dir, "embedding_data", "ami_curves_2d3d.npz")
np.savez(
    ami_npz_path,
    **{f"{ch}_ami": vals for ch, vals in ami_store.items()}
)
print(f"Saved AMI curves: {ami_npz_path}")

# -------------------------------------------------------
# ZIP EMBEDDINGS
# -------------------------------------------------------
for dir_path, zip_name in [
    (embedding_2d_dir, "2d_embedded_data.zip"),
    (embedding_3d_dir, "3d_embedded_data.zip")
]:
    zip_path = os.path.join(base_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        for file in sorted(os.listdir(dir_path)):
            full = os.path.join(dir_path, file)
            if os.path.isfile(full):
                zipf.write(full, arcname=file)
    print(f"Zipped: {zip_path}")

print("\nAll processes completed successfully.")

### 4D to 10D space reconstructions

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# -----------------------------
# CONFIG
# -----------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"
embedding_out_dir = os.path.join(base_dir, "embedding_data", "embeddings_4to10")
os.makedirs(embedding_out_dir, exist_ok=True)

eeg_path = os.path.join(base_dir, "eeg_data_with_channels.npy")

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

sampling_rate = 1000
start_time, end_time = 814.571, 921.515
start_index, end_index = int(start_time * sampling_rate), int(end_time * sampling_rate)

# Delay estimation settings
max_delay = 20
subsample_factor = 10
n_bins = 64
delay_rule = "first_local_min"   # "first_local_min" or "global_min"

# Embedding dimensions
emb_dims = list(range(4, 11))  # 4..10 inclusive

# -----------------------------
# LOAD EEG DATA
# -----------------------------
EEG_data = np.load(eeg_path, allow_pickle=True)

if EEG_data.ndim != 2:
    raise ValueError(f"Expected 2D EEG array, got shape {EEG_data.shape}")

# Handle either (time, channels) or (channels, time)
if EEG_data.shape[1] == len(eeg_channel_names):
    filtered_EEG_data = EEG_data[start_index:end_index, :]
elif EEG_data.shape[0] == len(eeg_channel_names):
    filtered_EEG_data = EEG_data[:, start_index:end_index].T
else:
    raise ValueError(f"Unexpected EEG shape: {EEG_data.shape}")

print(f"Loaded EEG segment: {filtered_EEG_data.shape}")
print(f"Segment: [{start_time}, {end_time}] sec")

# -----------------------------
# AMI / DELAY ESTIMATION
# -----------------------------
def standardize_1d(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + 1e-12)

def digitize_equal_width(x, n_bins=64):
    """
    Equal-width binning after standardization.
    Returns integer bin IDs in [0, n_bins-1].
    """
    x = standardize_1d(x)
    xmin, xmax = np.min(x), np.max(x)

    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
        return np.zeros_like(x, dtype=np.int32)

    edges = np.linspace(xmin, xmax, n_bins + 1)
    xb = np.digitize(x, edges[1:-1], right=False).astype(np.int32)
    xb = np.clip(xb, 0, n_bins - 1)
    return xb

def average_mutual_information_from_binned(xb, max_delay=20, n_bins=64):
    """
    Fast AMI using joint histograms via bincount.
    """
    xb = np.asarray(xb, dtype=np.int32)
    ami = np.full(max_delay, np.nan, dtype=float)

    for tau in range(1, max_delay + 1):
        x1 = xb[:-tau]
        x2 = xb[tau:]

        joint_idx = x1 * n_bins + x2
        joint = np.bincount(joint_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins).astype(float)

        total = joint.sum()
        if total <= 0:
            continue

        pxy = joint / total
        px = pxy.sum(axis=1, keepdims=True)
        py = pxy.sum(axis=0, keepdims=True)

        prod = px @ py
        mask = pxy > 0
        ami[tau - 1] = np.sum(pxy[mask] * np.log(pxy[mask] / prod[mask]))

    return ami

def first_local_minimum(y):
    y = np.asarray(y, dtype=float)
    for i in range(1, len(y) - 1):
        if np.isfinite(y[i-1]) and np.isfinite(y[i]) and np.isfinite(y[i+1]):
            if y[i] < y[i-1] and y[i] <= y[i+1]:
                return i
    return int(np.nanargmin(y))

def determine_delay(data, max_delay=20, subsample_factor=10, n_bins=64, rule="first_local_min"):
    xs = np.asarray(data, dtype=float)[::subsample_factor]
    xb = digitize_equal_width(xs, n_bins=n_bins)
    ami = average_mutual_information_from_binned(xb, max_delay=max_delay, n_bins=n_bins)

    if rule == "first_local_min":
        idx = first_local_minimum(ami)
    elif rule == "global_min":
        idx = int(np.nanargmin(ami))
    else:
        raise ValueError(f"Unknown rule: {rule}")

    tau = idx + 1
    return tau, ami

# -----------------------------
# DELAY EMBEDDING
# -----------------------------
def delay_embedding(data, emb_dim, delay):
    """
    Fast delay embedding using sliding_window_view.
    """
    x = np.asarray(data, dtype=float)
    span = (emb_dim - 1) * delay + 1
    M = len(x) - span + 1

    if M <= 0:
        return None

    win = sliding_window_view(x, span)
    emb = win[:, ::delay]
    return np.asarray(emb, dtype=float)

# -----------------------------
# MAIN LOOP: GENERATE 4D..10D
# -----------------------------
taus = {}
ami_store = {}
summary_rows = []

for ch_idx, channel_name in enumerate(eeg_channel_names):
    channel_data = filtered_EEG_data[:, ch_idx].astype(float)

    tau, ami_values = determine_delay(
        channel_data,
        max_delay=max_delay,
        subsample_factor=subsample_factor,
        n_bins=n_bins,
        rule=delay_rule
    )

    taus[channel_name] = tau
    ami_store[channel_name] = ami_values

    print(f"\n{channel_name:>3s} | tau={tau}")

    for emb_dim in emb_dims:
        embedded = delay_embedding(channel_data, emb_dim, tau)

        if embedded is None:
            print(f"  Skipping {channel_name} (m={emb_dim}): too short for tau={tau}")
            continue

        out_path = os.path.join(embedding_out_dir, f"{emb_dim}dembedded_{channel_name}.npy")
        np.save(out_path, embedded)

        summary_rows.append({
            "channel": channel_name,
            "tau": tau,
            "emb_dim": emb_dim,
            "n_points": embedded.shape[0],
            "ambient_dim": embedded.shape[1],
            "file": os.path.basename(out_path)
        })

        print(f"  Saved {emb_dim}D: {channel_name} | tau={tau} | shape={embedded.shape}")

print("\nEmbedding generation (4D..10D) completed.")

# -----------------------------
# SAVE TAUS FOR REPRODUCIBILITY
# -----------------------------
taus_npy_path = os.path.join(embedding_out_dir, "taus_4to10.npy")
np.save(taus_npy_path, taus, allow_pickle=True)

taus_csv_path = os.path.join(embedding_out_dir, "taus_4to10.csv")
pd.DataFrame(
    [{"channel": ch, "tau": tau} for ch, tau in taus.items()]
).to_csv(taus_csv_path, index=False)

print(f"Saved taus dict to: {taus_npy_path}")
print(f"Saved taus CSV to: {taus_csv_path}")

# -----------------------------
# SAVE AMI CURVES
# -----------------------------
ami_npz_path = os.path.join(embedding_out_dir, "ami_curves_4to10.npz")
np.savez(
    ami_npz_path,
    **{f"{ch}_ami": vals for ch, vals in ami_store.items()}
)
print(f"Saved AMI curves to: {ami_npz_path}")

# -----------------------------
# SAVE GENERATION SUMMARY
# -----------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv_path = os.path.join(embedding_out_dir, "embedding_generation_summary_4to10.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"Saved summary CSV to: {summary_csv_path}")

# -----------------------------
# ZIP OUTPUTS
# -----------------------------
zip_path = os.path.join(base_dir, "embedded_data_4to10.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for fname in sorted(os.listdir(embedding_out_dir)):
        full_path = os.path.join(embedding_out_dir, fname)
        if os.path.isfile(full_path):
            zipf.write(full_path, arcname=fname)

print(f"Zipped 4D..10D embeddings to: {zip_path}")